In [23]:
import pandas as pd
from datetime import datetime
from glob import glob

In [24]:
year = 2024
day = "FRI"

In [25]:
filelist = glob(f"/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/outputs/{year}/{day}_SPXW_DAY_OF_WEEK_DELTA_CONDOR_STRATEGY/consolidated_store/*.csv")
len(filelist)

249

In [26]:
df_list = [pd.read_csv(i) for i in filelist]


backtest_df = pd.concat(df_list, ignore_index=True)
backtest_df = backtest_df[backtest_df['trade_done']==True]
backtest_df['Timestamp'] = pd.to_datetime(backtest_df['timestamp'])
backtest_df = backtest_df.set_index("Timestamp")
backtest_df = backtest_df.sort_index()

In [27]:
backtest_df

,timestamp,trade_done,spot,atm_iv,portfolio_delta,portfolio_gamma,portfolio_vega,portfolio_theta,portfolio_sigma,portfolio_cash,instruments_mtm,portfolio_value,total_contracts,transaction_fees
Timestamp,,,,,,,,,,,,,,
2024-01-02 09:30:00,2024-01-02 09:30:00,True,4787.25,0.116758,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.0
2024-01-02 09:31:00,2024-01-02 09:31:00,True,4788.50,0.118555,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.0
2024-01-02 09:32:00,2024-01-02 09:32:00,True,4789.50,0.114084,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.0
2024-01-02 09:33:00,2024-01-02 09:33:00,True,4788.50,0.116350,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.0
2024-01-02 09:34:00,2024-01-02 09:34:00,True,4788.25,0.112462,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27 15:56:00,2024-12-27 15:56:00,True,6022.25,0.266757,0.0,0.0,0.0,0.0,0.0,237718.326,0.0,237718.326,0.0,0.0
2024-12-27 15:57:00,2024-12-27 15:57:00,True,6025.50,0.497243,0.0,0.0,0.0,0.0,0.0,237718.326,0.0,237718.326,0.0,0.0
2024-12-27 15:58:00,2024-12-27 15:58:00,True,6024.50,0.087990,0.0,0.0,0.0,0.0,0.0,237718.326,0.0,237718.326,0.0,0.0


In [28]:
daily_pnl = backtest_df.resample('D').last().dropna()
daily_pnl['daily_pnl'] = daily_pnl['portfolio_value'].diff()
daily_pnl['daily_pnl'].fillna(daily_pnl['portfolio_value'], inplace=True)
daily_pnl['abs_daily_pnl'] = abs(daily_pnl['daily_pnl'])

In [29]:
daily_pnl[['daily_pnl', 'abs_daily_pnl']].describe()

,daily_pnl,abs_daily_pnl
count,249.000000,249.000000
mean,954.692072,4554.963542
std,10230.571832,9205.887203
min,-57911.840000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,452.467000,5193.998000
max,42176.003000,57911.840000


In [30]:
date_list = daily_pnl['abs_daily_pnl'].nlargest(10).index.to_list()
date_list

[Timestamp('2024-12-27 00:00:00'),
 Timestamp('2024-11-15 00:00:00'),
 Timestamp('2024-04-04 00:00:00'),
 Timestamp('2024-08-02 00:00:00'),
 Timestamp('2024-11-01 00:00:00'),
 Timestamp('2024-12-19 00:00:00'),
 Timestamp('2024-04-05 00:00:00'),
 Timestamp('2024-12-20 00:00:00'),
 Timestamp('2024-09-27 00:00:00'),
 Timestamp('2024-10-11 00:00:00')]

In [31]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

def plot_dual_axis(df, x_col, y1_col, y2_col, y1_label='Primary Axis', y2_label='Secondary Axis'):
    fig = go.Figure()
    
    # Add trace for primary axis
    fig.add_trace(go.Scatter(x=df[x_col], y=df[y1_col], name=y1_label, yaxis='y1'))
    
    # Add trace for secondary axis
    fig.add_trace(go.Scatter(x=df[x_col], y=df[y2_col], name=y2_label, yaxis='y2'))
    
    # Update layout to add secondary axis
    fig.update_layout(
        title='Dual-Axis Plot with Plotly',
        xaxis=dict(title=x_col),
        yaxis=dict(title=y1_label, side='left', showgrid=False),
        yaxis2=dict(title=y2_label, side='right', overlaying='y', showgrid=False)
    )
    
    fig.show()
 

In [32]:
# import IPython
# IPython.get_ipython().magic("reset -sf")

In [33]:
# Call function
for date1 in date_list:
    print(date1.strftime("%Y-%m-%d"))
    day_df = backtest_df.loc[date1.strftime("%Y-%m-%d")]

    plot_dual_axis(day_df, 'timestamp', 'portfolio_value', 'atm_iv', 'pnl', 'iv')

2024-12-27


2024-11-15


2024-04-04


2024-08-02


2024-11-01


2024-12-19


2024-04-05


2024-12-20


2024-09-27


2024-10-11
